# ClusterSLM — Qwen2.5-Coder LoRA fine-tuning

Use a GPU accelerator. This notebook runs the included `train_lora.py` on chat-formatted JSONL. The default synthetic data is for smoke-testing only, not for meaningful security performance claims.


In [ ]:
%pip install -q 'transformers>=4.45,<5' 'accelerate>=0.34' 'peft>=0.13' 'bitsandbytes>=0.43' 'datasets>=3.0' 'safetensors>=0.4'


In [ ]:
from pathlib import Path
import glob, zipfile, os

# If you attached the full project ZIP as a Kaggle input, extract it.
ZIP_CANDIDATES = glob.glob('/kaggle/input/**/*.zip', recursive=True)
PROJECT_ROOT = None
for zpath in ZIP_CANDIDATES:
    try:
        with zipfile.ZipFile(zpath) as zf:
            names = zf.namelist()
            if any(n.endswith('train_lora.py') for n in names):
                zf.extractall('/kaggle/working/clusterSLM_source')
                matches = list(Path('/kaggle/working/clusterSLM_source').rglob('train_lora.py'))
                if matches:
                    PROJECT_ROOT = matches[0].parent
                    break
    except zipfile.BadZipFile:
        pass
if PROJECT_ROOT is None:
    matches = list(Path('/kaggle/input').rglob('train_lora.py'))
    if matches:
        PROJECT_ROOT = matches[0].parent
print('PROJECT_ROOT =', PROJECT_ROOT)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Attach the ClusterSLM project ZIP as a Kaggle input or make train_lora.py available under /kaggle/input.')

train_candidates = [p for p in Path('/kaggle/input').rglob('train.jsonl') if PROJECT_ROOT not in p.parents]
test_candidates = [p for p in Path('/kaggle/input').rglob('test.jsonl') if PROJECT_ROOT not in p.parents]
print('Train candidates:', train_candidates)
print('Test candidates:', test_candidates)
if not train_candidates:
    raise FileNotFoundError('Attach a dataset containing train.jsonl under /kaggle/input.')
TRAIN_FILE = str(train_candidates[0])
TEST_FILE = str(test_candidates[0]) if test_candidates else None
print('Using train:', TRAIN_FILE)
print('Using test:', TEST_FILE)


In [ ]:
import subprocess, sys

cmd = [
    sys.executable, str(PROJECT_ROOT / 'train_lora.py'),
    '--train-file', TRAIN_FILE,
    '--model-id', 'Qwen/Qwen2.5-Coder-1.5B-Instruct',
    '--output-dir', '/kaggle/working/clusterslm-qwen-lora',
    '--epochs', '2',
    '--batch-size', '1',
    '--grad-accum', '8',
    '--max-length', '2048',
]
if TEST_FILE:
    cmd += ['--test-file', TEST_FILE]
print('Running:', ' '.join(cmd))
subprocess.run(cmd, check=True, cwd=str(PROJECT_ROOT))


## Output and next steps

The adapter should be in `/kaggle/working/clusterslm-qwen-lora`. Download it from the notebook's Output/Files panel before the session ends. This is a LoRA adapter, not a standalone full model. Keep the base model ID `Qwen/Qwen2.5-Coder-1.5B-Instruct`.

For actual experiments, replace synthetic examples with reviewed, licensed code; split by repository/project rather than random snippets; and evaluate against a held-out labeled corpus. Training completion does not prove that the model detects vulnerabilities correctly.
